In [0]:
connection_string = (
    "Endpoint=sb://crypto-pipeline.servicebus.windows.net/;"
    "SharedAccessKeyName=send-policy;"
    "SharedAccessKey=xTiwXp9nlb34L4PyrxAbf0dqZ6/LsRRlr+AEhC5Rjzc=;"
    "EntityPath=crypto-stream"
)

event_hubs_conf = {
  "eventhubs.connectionString": connection_string
}

In [0]:
from pyspark.sql.functions import col, from_json

# 1) use LISTEN policy (the one you just pasted)
eh_conn_str = (
    "Endpoint=sb://crypto-pipeline.servicebus.windows.net/;"
    "SharedAccessKeyName=listen-policy;"
    "SharedAccessKey=f3N/RigtPBYdEvmZKnnNCTkMvfHIWGJk5+AEhHISouw=;"
    "EntityPath=crypto-stream"
)

# 2) encrypt for the connector
connection_string_encrypted = sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(eh_conn_str)

event_hubs_conf = {
    "eventhubs.connectionString": connection_string_encrypted
}

# 3) read stream from Event Hub
df_raw = (
    spark.readStream
         .format("eventhubs")
         .options(**event_hubs_conf)
         .load()
)

# 4) parse JSON from body
schema = """
  symbol STRING,
  price DOUBLE,
  ts_utc STRING,
  source STRING
"""

parsed_df = (
    df_raw
      .selectExpr("CAST(body AS STRING) AS json_str")
      .select(from_json(col("json_str"), schema).alias("data"))
      .select("data.*")
)

# 5) write to Delta (bronze)
bronze_path = "dbfs:/mnt/crypto/bronze/crypto_ticks"
checkpoint_path = "dbfs:/mnt/crypto/checkpoints/crypto_ticks"

query = (
    parsed_df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .start(bronze_path)
)

In [0]:
spark.read.format("delta") \
     .load("dbfs:/mnt/crypto/bronze/crypto_ticks") \
     .orderBy("ts_utc", ascending=False) \
     .show(truncate=False)

+------+-----------+--------------------------------+--------------------+
|symbol|price      |ts_utc                          |source              |
+------+-----------+--------------------------------+--------------------+
|LTC   |108.1304   |2025-11-10T06:48:02.683609+00:00|alpha_vantage_or_sim|
|XRP   |2.4077     |2025-11-10T06:48:02.622777+00:00|alpha_vantage_or_sim|
|SOL   |165.8638   |2025-11-10T06:48:02.565578+00:00|alpha_vantage_or_sim|
|ETH   |3641.6608  |2025-11-10T06:48:02.503908+00:00|alpha_vantage_or_sim|
|BTC   |106570.441 |2025-11-10T06:48:02.431246+00:00|alpha_vantage_or_sim|
|LTC   |108.1772   |2025-11-10T06:47:32.365603+00:00|alpha_vantage_or_sim|
|XRP   |2.4097     |2025-11-10T06:47:32.285823+00:00|alpha_vantage_or_sim|
|SOL   |165.717    |2025-11-10T06:47:32.226685+00:00|alpha_vantage_or_sim|
|ETH   |3637.0345  |2025-11-10T06:47:32.168853+00:00|alpha_vantage_or_sim|
|BTC   |106407.0342|2025-11-10T06:47:32.099385+00:00|alpha_vantage_or_sim|
|LTC   |107.9984   |2025-

In [0]:
from pyspark.sql.functions import to_timestamp, current_timestamp

bronze_path = "dbfs:/mnt/crypto/bronze/crypto_ticks"
silver_path = "dbfs:/mnt/crypto/silver/crypto_ticks"

bronze_stream = spark.readStream.format("delta").load(bronze_path)

silver_df = (
    bronze_stream
    .withColumn("event_time", to_timestamp("ts_utc"))   # parse string → timestamp
    .withColumn("ingest_time", current_timestamp())     # when we landed it
)

silver_query = (
    silver_df.writeStream
        .format("delta")
        .option("checkpointLocation", "dbfs:/mnt/crypto/checkpoints/crypto_ticks_silver")
        .outputMode("append")
        .start(silver_path)
)

In [0]:
spark.read.format("delta").load(silver_path).orderBy("event_time", ascending=False).show(truncate=False)

+------+-----------+--------------------------------+--------------------+--------------------------+-----------------------+
|symbol|price      |ts_utc                          |source              |event_time                |ingest_time            |
+------+-----------+--------------------------------+--------------------+--------------------------+-----------------------+
|LTC   |108.1772   |2025-11-10T06:47:32.365603+00:00|alpha_vantage_or_sim|2025-11-10 06:47:32.365603|2025-11-10 06:47:42.08 |
|XRP   |2.4097     |2025-11-10T06:47:32.285823+00:00|alpha_vantage_or_sim|2025-11-10 06:47:32.285823|2025-11-10 06:47:42.08 |
|SOL   |165.717    |2025-11-10T06:47:32.226685+00:00|alpha_vantage_or_sim|2025-11-10 06:47:32.226685|2025-11-10 06:47:42.08 |
|ETH   |3637.0345  |2025-11-10T06:47:32.168853+00:00|alpha_vantage_or_sim|2025-11-10 06:47:32.168853|2025-11-10 06:47:42.08 |
|BTC   |106407.0342|2025-11-10T06:47:32.099385+00:00|alpha_vantage_or_sim|2025-11-10 06:47:32.099385|2025-11-10 06:47:

In [0]:
from pyspark.sql.functions import window, first, max as spark_max, min as spark_min, last, avg, col

silver_path = "dbfs:/mnt/crypto/silver/crypto_ticks"
gold_path   = "dbfs:/mnt/crypto/gold/crypto_1m_bars"

silver_stream = spark.readStream.format("delta").load(silver_path)

# build 1-minute bars
bars_df = (
    silver_stream
      .withWatermark("event_time", "2 minutes")  # tolerate a bit of delay
      .groupBy(
          col("symbol"),
          window(col("event_time"), "1 minute").alias("w")
      )
      .agg(
          first("price").alias("open_price"),
          spark_max("price").alias("high_price"),
          spark_min("price").alias("low_price"),
          last("price").alias("close_price"),
          avg("price").alias("avg_price"),
      )
      .select(
          col("symbol"),
          col("w.start").alias("window_start"),
          col("w.end").alias("window_end"),
          "open_price",
          "high_price",
          "low_price",
          "close_price",
          "avg_price"
      )
)

bars_query = (
    bars_df.writeStream
        .format("delta")
        .option("checkpointLocation", "dbfs:/mnt/crypto/checkpoints/crypto_1m_bars")
        .outputMode("append")
        .start(gold_path)
)

In [0]:
spark.read.format("delta").load("dbfs:/mnt/crypto/gold/crypto_1m_bars").orderBy("window_start", ascending=False).show(truncate=False)

+------+-------------------+-------------------+-----------+-----------+-----------+-----------+------------------+
|symbol|window_start       |window_end         |open_price |high_price |low_price  |close_price|avg_price         |
+------+-------------------+-------------------+-----------+-----------+-----------+-----------+------------------+
|SOL   |2025-11-10 06:44:00|2025-11-10 06:45:00|164.714    |164.714    |164.714    |164.714    |164.714           |
|XRP   |2025-11-10 06:44:00|2025-11-10 06:45:00|2.4034     |2.4034     |2.4034     |2.4034     |2.4034            |
|LTC   |2025-11-10 06:44:00|2025-11-10 06:45:00|108.014    |108.014    |108.014    |108.014    |108.014           |
|BTC   |2025-11-10 06:44:00|2025-11-10 06:45:00|106173.997 |106173.997 |106173.997 |106173.997 |106173.997        |
|ETH   |2025-11-10 06:44:00|2025-11-10 06:45:00|3632.0697  |3632.0697  |3632.0697  |3632.0697  |3632.0697         |
|LTC   |2025-11-10 06:43:00|2025-11-10 06:44:00|108.1794   |108.1794   |